# Plot the Results of Model Evaluations

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from adjustText import adjust_text

from ultrabench.models.echocare.echocare_backbone import EchoCareBackbone
from ultrabench.models.samus.samus_backbone import SAMUSBackbone
from ultrabench.models.ultrasam.ultrasam_backbone import UltraSAMBackbone
from ultrabench.models.urfm.urfm_backbone import URFMBackbone
from ultrabench.models.usfm.usfm_backbone import USFMBackbone
from ultrabench.models.usfmae.usfmae_backbone import USFMAEBackbone

from notebook_utils import aggregate_test_metrics  # ty:ignore[unresolved-import]

# Set default figure size for all plots
plt.rcParams["figure.figsize"] = (3.25, 2.48)

SWEEPS_DIR = Path("/project/ultrabench2/runs/")

sns.set_theme(style="whitegrid", context="paper", palette="colorblind", font_scale=0.8)

## Parse Test Logs

Parse the `test_metrics.csv` files for the runs.

In [ ]:
test_results = aggregate_test_metrics(SWEEPS_DIR)

## Performance vs. Release Date

In [ ]:
release_dates = {
    "echocare": "2025-09-15",
    "samus": "2023-09-13",
    "ultrasam": "2024-11-25",
    "urfm": "2025-08-15",
    "usfm": "2023-12-30",
    "usfmae": "2025-10-27",
}

In [ ]:
# Group the classification results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_acc`
# for each group
classification_test_results = test_results[
    test_results["task"].str.contains("classification")
]

classification_test_summary = (
    classification_test_results
    .groupby(["task", "model", "shot_pct", "mode"])
    .agg({
        "f1_macro": ["mean"],
    })
    .reset_index()
)

# Flatten the multi-index columns
classification_test_summary.columns = [
    "task",
    "model",
    "shot_pct",
    "mode",
    "macro_f1",
]

classification_test_summary["release_date"] = classification_test_summary["model"].map(
    release_dates
)
classification_test_summary["release_date"] = pd.to_datetime(
    classification_test_summary["release_date"]
)

classification_test_summary.head()

In [ ]:
# Plot overall classification performance by release date
import matplotlib

plot_df = (
    classification_test_summary[
        (classification_test_summary["shot_pct"] == 1.0)
        & (classification_test_summary["mode"] == "linear_classifier_full_finetuning")
    ]
    .groupby(["model", "release_date"])
    .agg({"macro_f1": "mean"})
    .reset_index()
)

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="release_date",
    y="macro_f1",
    hue="model",
    ax=ax,
    legend=False,
)

# Add labels for each point
for _, row in plot_df.iterrows():
    ax.annotate(
        row["model"],
        (row["release_date"], row["macro_f1"]),
        textcoords="offset points",
        xytext=(0, 5),
        ha="center",
    )

# Draw a line connecting the current best model to the next best model
sorted_df = plot_df.sort_values(by="release_date")
current_best_model = sorted_df.iloc[0]
for i in range(1, len(sorted_df)):
    next_model = sorted_df.iloc[i]
    if next_model["macro_f1"] > current_best_model["macro_f1"]:
        ax.plot(
            [current_best_model["release_date"], next_model["release_date"]],
            [current_best_model["macro_f1"], next_model["macro_f1"]],
            color="gray",
            linestyle="--",
            linewidth=0.5,
        )
        current_best_model = next_model

ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Mean F1 Score")

x_labels = ax.get_xticklabels()
x_ticks = ax.get_xticks()
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, rotation=45, ha="right")
ax.set_xlabel("Release Date")

plt.tight_layout()
plt.savefig("classification_performance_by_release_date.pdf")
plt.show()

In [ ]:
# Group the segmentation results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_dice`
# for each group
segmentation_test_results = test_results[
    test_results["task"].str.contains("segmentation")
]

segmentation_test_summary = (
    segmentation_test_results
    .groupby(["task", "model", "shot_pct", "mode"])
    .agg({
        "positive_dice": ["mean"],
    })
    .reset_index()
)

# Flatten the multi-index columns
segmentation_test_summary.columns = [
    "task",
    "model",
    "shot_pct",
    "mode",
    "positive_dice",
]

segmentation_test_summary["release_date"] = segmentation_test_summary["model"].map(
    release_dates
)
segmentation_test_summary["release_date"] = pd.to_datetime(
    segmentation_test_summary["release_date"]
)

segmentation_test_summary.head()

In [ ]:
# Plot overall segmentation performance by release date
plot_df = (
    segmentation_test_summary[
        (segmentation_test_summary["shot_pct"] == 1.0)
        & (segmentation_test_summary["mode"].str.contains("full_finetuning"))
    ]
    .groupby(["model", "release_date"])
    .agg({"positive_dice": "mean"})
    .reset_index()
)

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="release_date",
    y="positive_dice",
    hue="model",
    ax=ax,
    legend=False,
)

# Add labels for each point
for _, row in plot_df.iterrows():
    ax.annotate(
        row["model"],
        (row["release_date"], row["positive_dice"]),
        textcoords="offset points",
        xytext=(0, 5),
        ha="center",
    )

# Draw a line connecting the current best model to the next best model
sorted_df = plot_df.sort_values(by="release_date")
current_best_model = sorted_df.iloc[0]
for i in range(1, len(sorted_df)):
    next_model = sorted_df.iloc[i]
    if next_model["positive_dice"] > current_best_model["positive_dice"]:
        ax.plot(
            [current_best_model["release_date"], next_model["release_date"]],
            [current_best_model["positive_dice"], next_model["positive_dice"]],
            color="gray",
            linestyle="--",
            linewidth=0.5,
        )
        current_best_model = next_model

ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Mean Positive Dice Score")

x_labels = ax.get_xticklabels()
x_ticks = ax.get_xticks()
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, rotation=45, ha="right")
ax.set_xlabel("Release Date")

plt.tight_layout()
plt.savefig("segmentation_performance_by_release_date.pdf")
plt.show()

## Performance vs. Model Size

In [ ]:
from pyexpat import model
from unittest.mock import patch

_BACKBONE_SPECS: list[tuple[str, type, dict]] = [
    (
        "echocare",
        EchoCareBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "samus",
        SAMUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": False},
    ),
    (
        "ultrasam",
        UltraSAMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "urfm",
        URFMBackbone,
        {
            "checkpoint_path": "dummy",
            "model_variant": "urfm_mae_vit_base_patch16",
            "output_intermediate_features": True,
        },
    ),
    (
        "usfm",
        USFMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "usfmae",
        USFMAEBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
]


model_parameter_counts = {}
for model_key, cls, kwargs in _BACKBONE_SPECS:
    with patch.object(cls, "_load_checkpoint", lambda self: None):
        backbone = cls(**kwargs)

    # Count the number of trainable parameters in the model (excluding
    # "decoder" parameters)
    trainable_parameters = sum(
        p.numel()
        for name, p in backbone.named_parameters()
        if p.requires_grad and "decoder" not in name
    )

    model_parameter_counts[model_key] = trainable_parameters

    del backbone

In [ ]:
# Plot the overall classification performance on the X-axis, the overall
# segmentation performance on the Y-axis, and the number of trainable
# parameters (excluding decoder parameters) as the size of the points.
plot_df = (
    test_results[
        (test_results["shot_pct"] == 1.0)
        & (test_results["mode"].str.contains("full_finetuning"))
    ]
    .groupby(["model"])
    .agg({
        "f1_macro": "mean",
        "positive_dice": "mean",
    })
    .reset_index()
)
plot_df["params"] = plot_df["model"].map(model_parameter_counts)
plot_df["params"] = round(plot_df["params"] / 1e6, 1)  # Convert to millions

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="f1_macro",
    y="positive_dice",
    size="params",
    hue="params",
    ax=ax,
    legend=True,
)

# Add labels for each point
for _, row in plot_df.iterrows():
    ax.annotate(
        row["model"],
        (row["f1_macro"], row["positive_dice"]),
        textcoords="offset points",
        xytext=(0, 5),
        ha="center",
    )

handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{float(label):.1f}M" for label in labels]
plt.legend(handles=handles, labels=new_labels)

ax.set_xlabel("Mean F1 Score")
ax.set_ylabel("Mean Positive Dice Score")

plt.tight_layout()
plt.savefig("performance_vs_model_size.pdf")
plt.show()
